In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
df = pd.read_csv('loan_approval_dataset.csv')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.describe()

In [ ]:
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df.columns = df.columns.str.strip()

In [ ]:
df['education'].unique()

In [ ]:
df['self_employed'].unique()

In [ ]:
df['loan_status'].unique()

In [ ]:
df['loan_status'] = df['loan_status'].map({
    'Approved': 1,
    'Rejected': 0
})

EDA

In [ ]:
sns.histplot(df['loan_status'], kde=True)

In [ ]:
plt.figure(figsize=(14, 10))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
)

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.head()

In [ ]:
sns.countplot(x=df['education'])

In [ ]:
sns.countplot(x=df['self_employed'])

CHECKING OUTLIERS

In [ ]:
num_features = df.select_dtypes(include='number')

q1 = num_features.quantile(0.25)
q3 = num_features.quantile(0.75)

IQR = q3 - q1

lower_bound = q1 - 1.5 * IQR
upper_bound = q3 + 1.5 * IQR

outlier_mask = (
    (num_features < lower_bound) |
    (num_features > upper_bound)
).any(axis=1)

df = df.loc[~outlier_mask].copy()

df = df.reset_index(drop=True)

DATA CLEANING

In [ ]:
df = df.loc[df['residential_assets_value'] >= 0]

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.drop('loan_id', axis=1, inplace=True)

In [ ]:
num_features.skew().sort_values()

In [ ]:
df.info()

TRAIN TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split

skewed_col = ['bank_asset_value','commercial_assets_value','residential_assets_value']

other_num_col = ['no_of_dependents','income_annum','loan_term',
                'cibil_score','luxury_assets_value','loan_amount']

oridnal_col_1 = ['self_employed']
oridnal_col_2 = ['education']

feature_col = skewed_col + other_num_col + oridnal_col_1 + oridnal_col_2

X = df[feature_col]
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ENCODING PIPELINE CREATION

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer

skewed_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

normal_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

ordinal_pipeline_1 = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['No','Yes']]))
])

ordinal_pipeline_2 = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Not Graduate', 'Graduate']]))
])

preprocessor = ColumnTransformer(transformers=[
    ('Skewed_Pipeline', skewed_pipeline, skewed_col),
    ('Numeric_Pipeline', normal_pipeline, other_num_col),
    ('Odinal_Pipeline_1', ordinal_pipeline_1, oridnal_col_1),
    ('Odinal_Pipeline_2', ordinal_pipeline_2, oridnal_col_2)
])

MODEL TRAINING

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('Logistic_Regression', LogisticRegression())
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

acc = accuracy_score(y_test,y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)
conf = confusion_matrix(y_test, y_pred_lr)
report = classification_report(y_test, y_pred_lr)

print('ACCURACY', round(acc * 100, 2))
print('F1_SCORE', round(f1 * 100, 2))
print('CONFUSION MATRIX', conf)
print('report', report)

In [156]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

models = {
    'RandomForestClassifier': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('RandomForestClassifier', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoostingClassifier': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('GradientBoostingClassifier', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTreeClassifier': Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('DecisionTreeClassifier', DecisionTreeClassifier(random_state=42))
    ]),
}

result = []
best_model = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred_mod = model.predict(X_test)
    acc = accuracy_score(y_test,y_pred_mod)
    f1 = f1_score(y_test, y_pred_mod)
    conf = confusion_matrix(y_test, y_pred_mod)
    report = classification_report(y_test, y_pred_mod)

    best_model.append(model)

    result.append({
        'Model':name,
        'Accuracy' : acc,
        'F1_Score' : f1,
        'Confusion Matrix' : conf,
        'Report' : report,
    })

rs = pd.DataFrame(result)


In [160]:
rs

,Model,Accuracy,F1_Score,Confusion Matrix,Report
0,RandomForestClassifier,0.980512,0.984436,"[[299, 10], [6, 506]]",precision recall f1-score ...
1,GradientBoostingClassifier,0.984166,0.987366,"[[300, 9], [4, 508]]",precision recall f1-score ...
2,DecisionTreeClassifier,0.981730,0.985451,"[[298, 11], [4, 508]]",precision recall f1-score ...


EXTRACTING BEST MODEL ON THE BASIS OF F1_SCORE

In [ ]:
best_idx = rs['F1_Score'].idxmax()
best_model_name = rs.loc[best_idx, 'Model']
best_model = models[best_model_name]

APPLYING RANDOMIZED CROSS VALIDATION ON BEST MODEL

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_list = {
    'GradientBoostingClassifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'GradientBoostingClassifier__n_estimators': [50, 100, 200, 300],
    'GradientBoostingClassifier__min_samples_split': [2, 5, 10],
    'GradientBoostingClassifier__min_samples_leaf': [1, 2, 4],
    'GradientBoostingClassifier__max_depth': [2, 5, 10],
}

random = RandomizedSearchCV(
    estimator=best_model,
    param_distributions=param_list,
    n_iter=15,
    cv=5,
    random_state=42,
    n_jobs=-1
)

random.fit(X_train, y_train)

y_pred_random = random.predict(X_test)

acc = accuracy_score(y_test,y_pred_random)
f1 = f1_score(y_test, y_pred_random)
conf = confusion_matrix(y_test, y_pred_random)
report = classification_report(y_test, y_pred_random)

print('ACCURACY', round(acc * 100, 2))
print('F1_SCORE', round(f1 * 100, 2))
print('CONFUSION MATRIX', conf)
print('report', report)

EXPORTING MODEL USING JOB_LIB

In [163]:
import joblib

joblib.dump(random.best_estimator_ , 'model.pkl')

['model.pkl']